## Environment (Carmack / UNE remote, NVIDIA L40)

Runs on the `carmack` server via VS Code Remote-SSH. Trained artifacts go to
persistent `/scratch/ttran72/`. This is a **PyTorch (timm)** notebook - make sure a
CUDA build of `torch` is installed in your venv (the earlier TensorFlow GPU setup
is a separate stack and does not cover torch).

**Running a long job safely (terminal, not the interactive kernel):**

Wrap the run in `tmux` so a dropped SSH session doesn't kill training, and `tee`
the output into `logs/` so you keep a record:

```bash
tmux new -s train                # detach with Ctrl-b then d; reattach: tmux attach -t train
source ~/venvs/phd/bin/activate
jupyter nbconvert --to notebook --execute --inplace \
    02-mobile-transformers.ipynb \
    2>&1 | tee /scratch/ttran72/logs/run_$(date +%F_%H%M).log
```

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('timm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', '--no-deps'])
import timm, torch
print('timm', timm.__version__, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  device:', torch.cuda.get_device_name(0))
    try:
        _ = (torch.zeros(1, 3, 4, 4, device='cuda') * 2).sum().item(); print('GPU kernel check: OK')
    except RuntimeError:
        print('GPU kernel check FAILED -> the torch CUDA build does not match the driver; reinstall torch, then rerun.'); raise
else:
    print('CUDA is NOT visible to PyTorch. Install a CUDA build of torch into this venv before training.')
    print('(A working TensorFlow GPU does NOT imply torch has CUDA - they ship separate CUDA runtimes.)')

## Config

In [ ]:
import os, torch, random, numpy as np
# --- sweep parameters: env overrides let run_grid.sh drive this headlessly ---
_ENV = os.environ.get
# ---- Carmack paths (persistent /scratch, survives between runs) ----
SCRATCH   = '/scratch/ttran72'
DATA_DIR  = f'{SCRATCH}/datasets/viet-medi-species-2026'   # ImageFolder root: one subfolder per class
CKPT_DIR  = f'{SCRATCH}/checkpoints'                        # checkpoints, .pt/.onnx, history csv, results json
CACHE_DIR = f'{SCRATCH}/cache'                              # split cache (lets reruns skip the ImageFolder rescan)
LOG_DIR   = f'{SCRATCH}/logs'                               # free for your own `tee` run logs (see below)
for _d in (CKPT_DIR, CACHE_DIR, LOG_DIR): os.makedirs(_d, exist_ok=True)
# -------------------------------------------------------------------
MODEL_NAMES  = ['mobilenetv2_100', 'mobilevit_xxs', 'efficientformerv2_s0']  # mobilenetv2_100 = same-protocol CNN baseline; comment out to fit a session
INPUT_SIZE   = 224    # divisible by 32 (160/192/224 valid; smaller = lighter)
BATCH_SIZE   = 64
VAL_SPLIT    = 0.15
TEST_SPLIT   = 0.15    # held-out test set; final reported metrics come from here
MIN_IMAGES_PER_CLASS = 25   # MUST match Notebook 1 (2,719-class >=25 filter)
EPOCHS       = int(_ENV('EPOCHS', '80'))   # high cap; early stopping (PATIENCE) decides the real stop, so the cap is not the limiter
LR           = 0.0001
WEIGHT_DECAY = 0.0001
WARMUP_EPOCHS   = 2
LABEL_SMOOTHING = 0.1
PATIENCE     = 7        # early-stop patience; now persisted across resumes
TOPK_WANTED  = [1, 5, 10]
NUM_WORKERS  = int(_ENV('NUM_WORKERS', '8'))   # data-loading bound at 8 (GPU ~24%); raise via env, e.g. NUM_WORKERS=16if RAM-bound
USE_AMP      = True
BALANCED_SAMPLER = False
RESUME       = True
SEED         = int(_ENV('SEED', '42'))     # vary across the grid for mean+/-std
# VIT_CEILING is auto-loaded from Notebook 1's output further below (after RUN_TAG).

# ---- Cesar's feature-highlighting preprocessing (MUST match Notebook 1) ----
PREPROCESS      = _ENV('PREP', 'baseline')   # 'baseline' | 'clahe' | 'sobel' | 'clahe_sobel'
SOURCE_DOWNSCALE = 1.0
# ----------------------------------------------------------------------------

RUN_TAG = PREPROCESS + ('' if SOURCE_DOWNSCALE == 1.0 else f'_ds{SOURCE_DOWNSCALE}') + f'_s{SEED}'
# ---- auto-load the ViT ceiling produced by Notebook 1 (no manual paste) ----
import glob as _glob, json as _json
def _load_vit_ceiling():
    exact = os.path.join(CKPT_DIR, f'vit_ceiling_{RUN_TAG}.json')
    cands = [exact] if os.path.exists(exact) else sorted(
        _glob.glob(os.path.join(CKPT_DIR, f'vit_ceiling_{PREPROCESS}_s*.json')))
    for _p in cands:
        try:    return _json.load(open(_p)).get('vit_ceiling_top1')
        except Exception: pass
    return None
VIT_CEILING = _load_vit_ceiling()
print('ViT ceiling:', VIT_CEILING if VIT_CEILING is not None
      else 'not found (run Notebook 1 for this preprocess/seed)')
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| preprocess:', PREPROCESS, '| seed:', SEED, '| epochs cap:', EPOCHS, '| run tag:', RUN_TAG)

# quick sanity check on the dataset root before the long ImageFolder walk
if os.path.isdir(DATA_DIR):
    _n = sum(1 for e in os.scandir(DATA_DIR) if e.is_dir())
    print(f'DATA_DIR OK: {DATA_DIR}  ({_n} class folders)')
    if _n <= 1:
        print('  WARNING: found <=1 class folder - the archive probably unzipped into a nested dir.')
        print('  Point DATA_DIR at the folder that directly contains the per-class subfolders.')
else:
    print(f'DATA_DIR MISSING: {DATA_DIR}')
    print('  -> download + unzip the Kaggle dataset into scratch first (see the markdown above).')

## Resume seed (for chained background commits)

In [ ]:
# On Carmack, /scratch persists between runs, so the Kaggle "seed the working dir
# from a previous commit's output" dance is unnecessary - RESUME reads checkpoints
# straight out of CKPT_DIR. Kept as a no-op so cell numbering still matches the paper.
RESUME_FROM = None   # normally leave None on Carmack
if RESUME_FROM:
    import shutil, glob
    for f in glob.glob(os.path.join(RESUME_FROM, '*_ckpt.pt')):
        dst = os.path.join(CKPT_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst); print('seeded', os.path.basename(f))
print('RESUME reads checkpoints from', CKPT_DIR)

In [ ]:
import os, math, time, copy, random
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from torchvision import datasets, transforms as T
from PIL import Image
from timm.data import resolve_model_data_config, create_transform

## Cesar's preprocessing transform\n\nPIL RGB in -> PIL RGB out, prepended to timm's pipeline. `'baseline'` + `SOURCE_DOWNSCALE=1.0` is a no-op. **Med Herb Lens must replicate whichever mode you ship**, or on-device accuracy won't match.

In [ ]:
import cv2

class CesarPreprocess:
    """Feature-highlighting preprocessing. PIL RGB in -> PIL RGB out."""
    def __init__(self, mode='baseline', downscale=1.0):
        self.mode, self.downscale = mode, downscale

    def __call__(self, img):
        arr = np.asarray(img.convert('RGB'))  # HWC RGB uint8
        if self.downscale and self.downscale != 1.0:
            h, w = arr.shape[:2]
            arr = cv2.resize(arr, (max(1, int(w * self.downscale)), max(1, int(h * self.downscale))),
                             interpolation=cv2.INTER_AREA)
        if self.mode in ('clahe', 'clahe_sobel'):
            lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
            arr = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
        if self.mode in ('sobel', 'clahe_sobel'):
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
            gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
            mag = cv2.magnitude(gx, gy)
            mag = np.clip(mag / (mag.max() + 1e-6) * 255.0, 0, 255).astype(np.uint8)
            if self.mode == 'sobel':
                arr = cv2.cvtColor(mag, cv2.COLOR_GRAY2RGB)          # pure edge map
            else:
                edges = cv2.cvtColor(mag, cv2.COLOR_GRAY2RGB)
                arr = cv2.addWeighted(arr, 0.7, edges, 0.3, 0.0)     # contrast + shape
        return Image.fromarray(arr)

def build_transforms(dc, mode, downscale):
    train_tf = create_transform(**dc, is_training=True)
    eval_tf  = create_transform(**dc, is_training=False)
    if mode == 'baseline' and (downscale in (None, 1.0)):
        return train_tf, eval_tf                       # exact original pipeline
    pre = CesarPreprocess(mode=mode, downscale=downscale)
    return T.Compose([pre, *train_tf.transforms]), T.Compose([pre, *eval_tf.transforms])

In [ ]:
from sklearn.metrics import f1_score

class PlantDS(Dataset):
    def __init__(self, samples, tf): self.samples, self.tf = samples, tf
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, y = self.samples[i]
        return self.tf(Image.open(path).convert('RGB')), y

def filter_min_images(samples, class_names, min_n):
    """Keep only classes with >= min_n images; relabel kept classes to 0..C-1."""
    counts = Counter(y for _, y in samples)
    keep = sorted(y for y, c in counts.items() if c >= min_n)
    remap = {old: new for new, old in enumerate(keep)}
    new_samples = [(p, remap[y]) for p, y in samples if y in remap]
    new_class_names = [class_names[y] for y in keep]
    return new_samples, new_class_names

def stratified_split_3way(samples, val_split, test_split, seed):
    """Per-class stratified train/val/test. Every class contributes to all three
    splits (guaranteed >=1 train sample), so val/test cover the full class set."""
    by = {}
    for i, (p, y) in enumerate(samples):
        by.setdefault(y, []).append(i)
    rng = random.Random(seed); tr, va, te = [], [], []
    for y, ids in by.items():
        rng.shuffle(ids); n = len(ids)
        n_test = max(1, int(round(n * test_split)))
        n_val  = max(1, int(round(n * val_split)))
        if n_test + n_val >= n:                      # keep >=1 for training
            n_test = min(n_test, max(1, n - 2))
            n_val  = max(1, n - n_test - 1)
        te += ids[:n_test]; va += ids[n_test:n_test + n_val]; tr += ids[n_test + n_val:]
    pick = lambda idx: [samples[i] for i in idx]
    return pick(tr), pick(va), pick(te)

def topk_hits(logits, y, ks):
    top = logits.topk(max(ks), 1).indices
    correct = (top == y.unsqueeze(1))
    return {k: correct[:, :k].any(1).float().sum().item() for k in ks}

def make_loaders(train_s, val_s, train_tf, eval_tf, batch, workers, balanced):
    if balanced:
        counts = Counter(y for _, y in train_s)
        w = [1.0 / counts[y] for _, y in train_s]
        sampler = WeightedRandomSampler(w, num_samples=len(train_s), replacement=True)
        tl = DataLoader(PlantDS(train_s, train_tf), batch_size=batch, sampler=sampler,
                        num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                        prefetch_factor=(4 if workers > 0 else None))
    else:
        tl = DataLoader(PlantDS(train_s, train_tf), batch_size=batch, shuffle=True,
                        num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                        prefetch_factor=(4 if workers > 0 else None))
    vl = DataLoader(PlantDS(val_s, eval_tf), batch_size=batch, shuffle=False,
                    num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                        prefetch_factor=(4 if workers > 0 else None))
    return tl, vl

def make_eval_loader(samples, tf, batch, workers):
    return DataLoader(PlantDS(samples, tf), batch_size=batch, shuffle=False,
                      num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                        prefetch_factor=(4 if workers > 0 else None))

@torch.no_grad()
def evaluate(model, loader, ks, device, amp):
    """Final held-out evaluation: Top-k accuracy + macro/weighted F1 (top-1)."""
    model = model.to(device).eval()
    hits = {k: 0 for k in ks}; n = 0; y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with autocast('cuda', enabled=amp, dtype=torch.float16):
            logits = model(x).float().cpu()
        top = logits.topk(max(ks), 1).indices
        correct = (top == y.unsqueeze(1))
        for k in ks: hits[k] += correct[:, :k].any(1).float().sum().item()
        n += y.size(0)
        y_true.append(y.numpy()); y_pred.append(top[:, 0].numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
    accs = {k: hits[k] / n for k in ks}
    return {'accs': accs, 'n': int(n),
            'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0))}

def cosine_warmup(opt, warmup_steps, total_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


In [ ]:
def train_one(model, name, train_s, val_s, train_tf, eval_tf, cfg):
    device = cfg['device']
    print(f'  building dataloaders (workers={cfg["workers"]}) ...', flush=True)
    tl, vl = make_loaders(train_s, val_s, train_tf, eval_tf,
                          cfg['batch'], cfg['workers'], cfg['balanced'])
    print('  dataloaders ready', flush=True)
    for p in model.parameters(): p.requires_grad = True
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    steps = len(tl)
    sched = cosine_warmup(opt, cfg['warmup_ep'] * steps, cfg['epochs'] * steps)
    crit = nn.CrossEntropyLoss(label_smoothing=cfg['label_smooth'])
    scaler = GradScaler('cuda', enabled=cfg['amp'])
    ks = cfg['topk']
    ckpt_path = os.path.join(cfg['ckpt_dir'], f"{name}_{cfg['run_tag']}_ckpt.pt")
    hist_path = os.path.join(cfg['ckpt_dir'], f"{name}_{cfg['run_tag']}_history.csv")
    if not os.path.exists(hist_path):
        with open(hist_path, 'w') as _h:
            _h.write('model,run_tag,seed,epoch,train_loss,' + ','.join(f'val_top{k}' for k in ks) + ',epoch_seconds\n')

    # best_accs snapshots ALL Top-k at the improving epoch (fixes last-epoch reporting bug)
    # patience is persisted so early stopping behaves identically across chained resumes
    best_acc, best_state, best_accs, patience, start = 0.0, None, None, 0, 0
    total_train_s = 0.0
    if cfg['resume'] and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        start = ck['epoch'] + 1; best_acc = ck['best_acc']; best_state = ck['best_state']
        best_accs = ck.get('best_accs'); patience = ck.get('patience', 0)
        total_train_s = ck.get('total_train_s', 0.0)
        print(f'resumed {name} from epoch {start} (best Top-1={best_acc:.3f}, patience={patience})', flush=True)

    print(f'  entering training loop at epoch {start+1}/{cfg["epochs"]} ...', flush=True)
    for ep in range(start, cfg['epochs']):
        model.train(); run = 0.0; t0 = time.time()
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=cfg['amp'], dtype=torch.float16):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * x.size(0)
        model.eval(); hits = {k: 0 for k in ks}; n = 0
        with torch.no_grad(), autocast('cuda', enabled=cfg['amp'], dtype=torch.float16):
            for x, y in vl:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                h = topk_hits(model(x).float(), y, ks)
                for k in ks: hits[k] += h[k]
                n += x.size(0)
        accs = {k: hits[k] / n for k in ks}; top1 = accs[ks[0]]
        dt = time.time() - t0
        total_train_s += dt
        with open(hist_path, 'a') as _h:
            _h.write(f"{name},{cfg['run_tag']},{cfg.get('seed','')},{ep+1},{run/len(train_s):.6f},"
                     + ','.join(f'{accs[k]:.6f}' for k in ks) + f',{dt:.2f}\n')
        improved = top1 > best_acc
        if improved:
            best_acc, best_state, patience = top1, copy.deepcopy(model.state_dict()), 0
            best_accs = dict(accs)          # snapshot all Top-k at the best epoch
        else:
            patience += 1
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                    'best_acc': best_acc, 'best_state': best_state,
                    'best_accs': best_accs, 'patience': patience,
                    'total_train_s': total_train_s}, ckpt_path)
        msg = '  '.join(f'top{k}={accs[k]:.3f}' for k in ks)
        star = '  *best*' if improved else ''
        print(f'  epoch {ep+1:2d}/{cfg["epochs"]}  loss={run/len(train_s):.3f}  {msg}  '
              f'({dt:.0f}s/ep, ETA {dt*(cfg["epochs"]-ep-1)/60:.0f}m){star}', flush=True)
        if patience >= cfg['patience']:
            print(f'  early stop: no Top-1 gain for {cfg["patience"]} epochs'); break

    model.load_state_dict(best_state)
    return {'model': model.to('cpu').eval(),
            'params': sum(p.numel() for p in model.parameters()),
            'best_acc': best_acc, 'best_accs': best_accs,
            'runtime_s': total_train_s}   # best-epoch Top-k + train wall-clock

## Read dataset + stratified split (shared by both models)

In [ ]:
import pickle, time
# Cache the filtered + split sample lists so chained commits skip the ~15 min
# ImageFolder walk and reuse the EXACT same train/val/test split every run.
# The cache is invalidated automatically if DATA_DIR / filter / split fracs / seed change.
# split depends only on SEED (not preprocess), so key the cache by seed -> all
# preprocessing variants at a given seed reuse ONE scan and the IDENTICAL split.
cache_path = os.path.join(CACHE_DIR, f'split_cache_s{SEED}.pkl')
_key = dict(data_dir=DATA_DIR, min_imgs=MIN_IMAGES_PER_CLASS,
            val=VAL_SPLIT, test=TEST_SPLIT, seed=SEED)

cache = None
if os.path.exists(cache_path):
    with open(cache_path, 'rb') as f:
        cand = pickle.load(f)
    if cand.get('key') == _key:
        cache = cand
    else:
        print('split cache found but parameters changed -> rebuilding'); print('  cached:', cand.get('key')); print('  now   :', _key)

if cache is not None:
    CLASS_NAMES = cache['class_names']; train_s, val_s, test_s = cache['train'], cache['val'], cache['test']
    NUM_CLASSES = len(CLASS_NAMES)
    TOPK = [k for k in TOPK_WANTED if k <= NUM_CLASSES] or [1]
    if 1 not in TOPK: TOPK = [1] + TOPK
    print(f'loaded cached split (no rescan): {NUM_CLASSES} classes | '
          f'train/val/test {len(train_s)} / {len(val_s)} / {len(test_s)}')
    print('reporting Top-k:', TOPK)
else:
    _t0 = time.time()
    full = datasets.ImageFolder(DATA_DIR)
    samples_all, class_names_all = full.samples, full.classes
    samples, CLASS_NAMES = filter_min_images(samples_all, class_names_all, MIN_IMAGES_PER_CLASS)
    NUM_CLASSES = len(CLASS_NAMES)
    TOPK = [k for k in TOPK_WANTED if k <= NUM_CLASSES] or [1]
    if 1 not in TOPK: TOPK = [1] + TOPK

    counts = Counter(y for _, y in samples)
    c = np.array(sorted(counts.values()))
    print(f'raw classes: {len(class_names_all)}   ->   kept (>= {MIN_IMAGES_PER_CLASS} imgs/class): {NUM_CLASSES}')
    print(f'images after filter: {len(samples)}   (dropped {len(samples_all) - len(samples)})')
    print(f'images/class  min={c.min()}  median={int(np.median(c))}  max={c.max()}')

    train_s, val_s, test_s = stratified_split_3way(samples, VAL_SPLIT, TEST_SPLIT, SEED)
    print('train/val/test images:', len(train_s), '/', len(val_s), '/', len(test_s))
    print('reporting Top-k:', TOPK)

    with open(cache_path, 'wb') as f:
        pickle.dump({'key': _key, 'class_names': CLASS_NAMES,
                     'train': train_s, 'val': val_s, 'test': test_s}, f)
    print(f'cached split -> {cache_path}  (scan took {time.time()-_t0:.0f}s; future runs skip it)')


## Preview the preprocessing (confirm it matches Notebook 1)

In [ ]:
# preview only - must NEVER abort a headless run
try:
    import matplotlib
    matplotlib.use('Agg')          # headless-safe backend (no display under nbconvert)
    import matplotlib.pyplot as plt
    # train_s exists whether Cell 12 rebuilt the split or loaded it from cache
    # (unlike `samples`, which only exists on a fresh scan). Preview first train image.
    _pre = CesarPreprocess(mode=PREPROCESS, downscale=SOURCE_DOWNSCALE)
    _p = train_s[0][0]
    _orig = Image.open(_p).convert('RGB')
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(_orig); ax[0].set_title('source'); ax[0].axis('off')
    ax[1].imshow(_pre(_orig)); ax[1].set_title(f'PREPROCESS={PREPROCESS}'); ax[1].axis('off')
    plt.tight_layout()
    _prev_path = os.path.join(LOG_DIR, f'preprocess_preview_{RUN_TAG}.png')
    plt.savefig(_prev_path, dpi=110, bbox_inches='tight'); plt.close(fig)
    print('preprocess preview saved ->', _prev_path)
except Exception as _e:
    print('[preview skipped]', repr(_e)[:200])


## Train each mobile transformer

In [ ]:
import os
results = {}
for name in MODEL_NAMES:
    print('=' * 60); print(f'{name}  [{RUN_TAG}]'); print('=' * 60)
    _ckpt = os.path.join(CKPT_DIR, f'{name}_{RUN_TAG}_ckpt.pt')
    _resuming = RESUME and os.path.exists(_ckpt)
    print(f'creating {name} (pretrained={not _resuming}, resuming={_resuming}) ...', flush=True)
    model = timm.create_model(name, pretrained=not _resuming, num_classes=NUM_CLASSES)
    print('model ready', flush=True)
    dc = resolve_model_data_config(model); dc['input_size'] = (3, INPUT_SIZE, INPUT_SIZE)
    train_tf, eval_tf = build_transforms(dc, PREPROCESS, SOURCE_DOWNSCALE)
    cfg = dict(device=device, batch=BATCH_SIZE, workers=NUM_WORKERS, balanced=BALANCED_SAMPLER,
               lr=LR, wd=WEIGHT_DECAY, warmup_ep=WARMUP_EPOCHS, epochs=EPOCHS,
               label_smooth=LABEL_SMOOTHING, amp=USE_AMP, topk=TOPK, patience=PATIENCE,
               resume=RESUME, ckpt_dir=CKPT_DIR, run_tag=RUN_TAG, seed=SEED)
    r = train_one(model, name, train_s, val_s, train_tf, eval_tf, cfg)
    test_loader = make_eval_loader(test_s, eval_tf, BATCH_SIZE, NUM_WORKERS)
    r['test'] = evaluate(r['model'], test_loader, TOPK, device, USE_AMP)
    results[name] = r
    t = r['test']
    kmsg = '  '.join(f'top{k}={t["accs"][k]:.4f}' for k in TOPK)
    print(f'{name}: val_top1={r["best_acc"]:.4f} | TEST {kmsg} '
          f'macroF1={t["macro_f1"]:.4f} weightedF1={t["weighted_f1"]:.4f} '
          f'| params={r["params"]:,} | train {r.get("runtime_s", 0) / 60:.1f} min')

# ---- save metrics immediately (BEFORE any export), so a failed TFLite
#      conversion can never cost us the results of a full training run ----
import json as _json, platform as _plat
def _early_record(name, r):
    t = r.get('test', {})
    return {'model': name, 'run_tag': RUN_TAG, 'preprocess': PREPROCESS, 'seed': SEED,
            'num_classes': NUM_CLASSES, 'min_images_per_class': MIN_IMAGES_PER_CLASS,
            'split': {'train': len(train_s), 'val': len(val_s), 'test': len(test_s),
                      'val_frac': VAL_SPLIT, 'test_frac': TEST_SPLIT},
            'params': r.get('params'), 'val_top1_best': r.get('best_acc'),
            'test': {'n': t.get('n'),
                     **{f'top{k}': t.get('accs', {}).get(k) for k in TOPK},
                     'macro_f1': t.get('macro_f1'), 'weighted_f1': t.get('weighted_f1')},
            'train_runtime_s': r.get('runtime_s'),
            'history_csv': f'{name}_{RUN_TAG}_history.csv',
            'env': {'torch': torch.__version__,
                    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
                    'python': _plat.python_version()}}
_early = {name: _early_record(name, r) for name, r in results.items()}
_early_path = os.path.join(CKPT_DIR, f'results_{RUN_TAG}.json')
with open(_early_path, 'w') as _f:
    _json.dump(_early, _f, indent=2)
print('metrics saved (pre-export) ->', _early_path)

## Save weights + ONNX export

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'onnxscript'])
for name in MODEL_NAMES:
    m = results[name]['model']
    torch.save(m.state_dict(), f'{CKPT_DIR}/{name}_{RUN_TAG}.pt')
    dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
    try:
        torch.onnx.export(m, dummy, f'{CKPT_DIR}/{name}_{RUN_TAG}.onnx', opset_version=17,
                          input_names=['input'], output_names=['logits'], dynamo=False)
        print(f'{name}: ONNX saved')
    except Exception as e:
        print(f'{name}: ONNX export failed: {repr(e)[:150]}')

## ONNX -> TFLite (FP16 + FP32)\n\nIf this throws a numpy/import error, restart the kernel and run only this cell + the next (they read the .onnx files from disk).

In [ ]:
import subprocess, sys, os
# onnx2tf needs several small helper packages that are NOT pulled in automatically.
# sng4onnx was the missing one; the rest are its usual companions.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnx2tf', 'onnx', 'onnxruntime', 'onnx-graphsurgeon',
                'sng4onnx', 'onnxslim', 'ai-edge-litert'])
# NOTE: no kernel restart and no protobuf force-downgrade here - both break a
# headless `nbconvert` run and can destabilise the working torch/TF stack.

try:
    MODEL_NAMES, results, RUN_TAG
except NameError:
    MODEL_NAMES = ['mobilevit_xxs', 'efficientformerv2_s0']; RUN_TAG = 'baseline'
    results = {n: {} for n in MODEL_NAMES}

for name in MODEL_NAMES:
    tf_dir = f'{CKPT_DIR}/{name}_{RUN_TAG}_tf'
    onnx_path = f'{CKPT_DIR}/{name}_{RUN_TAG}.onnx'
    if not os.path.exists(onnx_path):
        print(f'{name}: no ONNX file ({onnx_path}), skipping conversion'); continue
    try:
        subprocess.run(['onnx2tf', '-i', onnx_path, '-o', tf_dir, '-n'], check=True)
        for f in os.listdir(tf_dir):
            if f.endswith('_float16.tflite'):
                results[name]['tflite_fp16_mb'] = round(os.path.getsize(os.path.join(tf_dir, f)) / 1e6, 2)
            elif f.endswith('_float32.tflite'):
                results[name]['tflite_fp32_mb'] = round(os.path.getsize(os.path.join(tf_dir, f)) / 1e6, 2)
        print(f"{name}: fp16={results[name].get('tflite_fp16_mb','-')} MB  "
              f"fp32={results[name].get('tflite_fp32_mb','-')} MB")
    except Exception as e:
        print(f'{name}: conversion failed (non-fatal): {repr(e)[:200]}')

## Parity check: PyTorch vs TFLite (FP32)\n\nChecks top-k ranking agreement on one fixed input. A top-1 mismatch or large delta means a transpose/attention op didn't lower cleanly.

In [ ]:
import glob, numpy as np, torch
try:
    from ai_edge_litert.interpreter import Interpreter
except Exception as _e:
    print('ai_edge_litert unavailable, skipping parity check (non-fatal):', repr(_e)[:120]); Interpreter = None

TOL_TOPK = 10

for name in ([] if Interpreter is None else MODEL_NAMES):
    tfl = glob.glob(f'{CKPT_DIR}/{name}_{RUN_TAG}_tf/*_float32.tflite')
    if not tfl:
        print(f'{name}: no fp32 tflite found, skipping parity'); continue

    torch.manual_seed(0)
    x = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)   # NCHW for torch

    m = results[name]['model'].eval()
    with torch.no_grad():
        pt_logits = m(x).float().numpy().ravel()

    it = Interpreter(model_path=tfl[0]); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    xin = x.numpy() if inp['shape'][-1] != 3 else x.permute(0, 2, 3, 1).numpy()  # onnx2tf emits NHWC
    it.set_tensor(inp['index'], xin.astype(inp['dtype']))
    it.invoke()
    tf_logits = it.get_tensor(out['index']).astype(np.float32).ravel()

    pt_top = pt_logits.argsort()[::-1][:TOL_TOPK]
    tf_top = tf_logits.argsort()[::-1][:TOL_TOPK]
    agree = int((pt_top == tf_top).sum())
    max_abs = float(np.max(np.abs(pt_logits - tf_logits)))
    results[name]['parity_top1_ok'] = bool(pt_top[0] == tf_top[0])
    print(f'{name}: top-{TOL_TOPK} agree {agree}/{TOL_TOPK}  '
          f'top-1 {"OK" if pt_top[0]==tf_top[0] else "MISMATCH"}  max|dlogit|={max_abs:.4f}')
    if pt_top[0] != tf_top[0]:
        print('  WARNING: top-1 disagrees - inspect NCHW/NHWC transpose or attention ops '
              'before trusting on-device numbers.')

## Summary table

In [ ]:
def _acc(r, k): return r['test']['accs'].get(k, 0)
print(f'preprocess = {RUN_TAG}    (all accuracy/F1 figures are on the HELD-OUT TEST set)')
hdr = (f'{"model":26s}{"params":>12s}' + ''.join(f'top{k}'.rjust(9) for k in TOPK)
       + f'{"macroF1":>9s}{"wF1":>8s}{"fp16MB":>8s}{"parity":>8s}{"min":>7s}')
print(hdr); print('-' * len(hdr))
if VIT_CEILING is not None:
    print(f'{"ViT-B/16 (ceiling)":26s}{"-":>12s}{VIT_CEILING:9.4f}'
          + '-'.rjust(9) * (len(TOPK) - 1) + f'{"-":>9}{"-":>8}{"-":>8}{"-":>8}{"-":>7}')
for name in MODEL_NAMES:
    r = results[name]; t = r.get('test', {})
    if not t: print(f'{name:26s}(no test metrics)'); continue
    kv = ''.join(f'{t["accs"].get(k, 0):9.4f}' for k in TOPK)
    par = {True: 'OK', False: 'FAIL'}.get(r.get('parity_top1_ok'), '-')
    print(f'{name:26s}{r.get("params", 0):>12,}{kv}{t.get("macro_f1", 0):9.4f}{t.get("weighted_f1", 0):8.4f}'
          f'{str(r.get("tflite_fp16_mb", "-")):>8}{par:>8}{r.get("runtime_s", 0) / 60:7.1f}')
print('\nTEST-set table: accuracy-vs-size frontier for the cloud-to-edge benchmark.')
print('Next: run per PREPROCESS mode for the sweep, then take the .tflite files to the Realme.')


## Persist results (JSON + CSV) for manuscript tables/figures

Writes `results_<run_tag>.json` (one self-describing record per model) and relies on the per-epoch `*_history.csv` files saved during training. Re-run this per `PREPROCESS` mode / seed; the JSON filename is tagged so runs don't overwrite each other. Attach these files as the notebook output to regenerate tables and curves without retraining.

In [ ]:
# ---- persist machine-readable results for the manuscript ----
# Writes results_<run_tag>.json (one record per model) so tables/figures can be
# regenerated later WITHOUT re-running training. Per-epoch curves are in the
# *_history.csv files written during training.
import json, os, glob, platform, torch

def _record(name, r):
    t = r.get('test', {})
    return {
        'model': name,
        'run_tag': RUN_TAG,
        'preprocess': PREPROCESS,
        'seed': SEED,
        'num_classes': NUM_CLASSES,
        'min_images_per_class': MIN_IMAGES_PER_CLASS,
        'split': {'train': len(train_s), 'val': len(val_s), 'test': len(test_s),
                  'val_frac': VAL_SPLIT, 'test_frac': TEST_SPLIT},
        'params': r.get('params'),
        'val_top1_best': r.get('best_acc'),
        'test': {
            'n': t.get('n'),
            **{f'top{k}': t.get('accs', {}).get(k) for k in TOPK},
            'macro_f1': t.get('macro_f1'),
            'weighted_f1': t.get('weighted_f1'),
        },
        'tflite_fp16_mb': r.get('tflite_fp16_mb'),
        'tflite_fp32_mb': r.get('tflite_fp32_mb'),
        'parity_top1_ok': r.get('parity_top1_ok'),
        'train_runtime_s': r.get('runtime_s'),
        'history_csv': f'{name}_{RUN_TAG}_history.csv',
        'env': {'torch': torch.__version__,
                'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
                'python': platform.python_version()},
    }

records = {name: _record(name, r) for name, r in results.items()}
out_path = os.path.join(CKPT_DIR, f'results_{RUN_TAG}.json')
with open(out_path, 'w') as f:
    json.dump(records, f, indent=2)

print('wrote', out_path)
print('history csv files:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(CKPT_DIR, '*_history.csv'))))
print(json.dumps(records, indent=2)[:1500])
